In [1]:
import pandas as pd
import sys
sys.path.append("..")

from features import extract_features, FEATURE_COLUMNS

df = pd.read_csv("../data/cleaned_malicious_phish.csv")   

df_features = extract_features(df)
df_features[FEATURE_COLUMNS]

label_map = {"benign": 0, "phishing": 1, "malware": 1, "defacement": 1}
df["label"] = df["type"].map(label_map)

In [2]:
import tldextract
from sklearn.model_selection import GroupShuffleSplit
def get_domain(url):
    ext = tldextract.extract(url)
    return f"{ext.domain}.{ext.suffix}"

df["domain"] = df["url"].apply(get_domain)

X = df_features[FEATURE_COLUMNS]
y = df["label"]
groups = df["domain"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in gss.split(X,y,groups):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]
    
print(X_train.shape, X_test.shape)


(519881, 4) (121244, 4)


In [3]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42)

scores = cross_val_score(model, X_train, y_train, cv=5, scoring="f1")

print(scores)
print(scores.mean())




[0.03853783 0.03772944 0.03779572 0.45350136 0.18044807]
0.14960248440451038


In [4]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

balanced_model = LogisticRegression(random_state=42, class_weight="balanced")

balanced_scores = cross_val_score(balanced_model, X_train, y_train, cv=5, scoring="f1")

print(balanced_scores)
print(balanced_scores.mean())




[0.60701252 0.60616525 0.60380744 0.64979074 0.63763887]
0.6208829641583933


## Finding

The same model family, with class weighting to counter the 2:1 imbalance, lifted cross-validated f1 from 015 to 0.62 and eliminated the fold-to-fold instability.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

rf_model = RandomForestClassifier(random_state=42, class_weight="balanced")

rf_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring="f1")

print(rf_scores)
print(rf_scores.mean())


[0.62406363 0.62571254 0.62434523 0.47442079 0.54996032]
0.5797005029499042


In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score

hgb_model = HistGradientBoostingClassifier(random_state=42, class_weight="balanced")

hgb_scores = cross_val_score(hgb_model, X_train, y_train, cv=5, scoring="f1")

print(hgb_scores)
print(hgb_scores.mean())


[0.62671552 0.62401671 0.62300952 0.62571271 0.56038186]
0.6119672651161807


## Finding

Complexity isn't automatically better; a well-configured simple model beat a random forest on this feature set. Across three families, the balanced logistic regression won (f1 0.62); tree-based models did not beat it, indicating the feature set's signal is largely additive rather than interaction-driven. 

In [7]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42, class_weight="balanced")

param_grid = {"C": [0.01, 0.1, 1, 10, 100]}

grid = GridSearchCV(model, param_grid, cv=5, scoring="f1")

grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)

{'C': 0.01}
0.6209826456790248


## Finding

Grid search over C selected 0.01 but improved f1 by ~0.0001 over the default; the model was already near-optmal, consistent with a small, additive feature set.